# 🦴 Bone Fracture Detection — YOLOv8l Training v2 (Improved)

**Changes from v1:**
- ✅ Upgraded to **YOLOv8l** (larger model, more capacity)
- ✅ **300 epochs** with `patience=50` (no more premature stopping)
- ✅ **AdamW optimizer** with warmup (better convergence vs SGD default)
- ✅ Lower initial LR (`lr0=0.001`) with cosine decay to `lrf=0.0001`
- ✅ **Stronger augmentation**: mixup, copy-paste, scale jitter, perspective
- ✅ **Image size 800** (better detection of small fracture features)
- ✅ **Label smoothing** + **close mosaic** tail-off
- ✅ **Auto batch size** (uses maximum GPU memory safely)
- ✅ Class weight hints via `cls` loss weight

**Dataset:** BoneFractureYolo8 (7-class detection)  
**Model:** YOLOv8l (pre-trained on COCO)  
**GPU:** Kaggle T4/P100

## Phase 0 — Environment Verification

In [ ]:
import subprocess, torch, os

print('=' * 60)
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'nvidia-smi not found')

print('=' * 60)
print(f'PyTorch version  : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is NOT available. Enable GPU accelerator in Kaggle.')

num_gpus = torch.cuda.device_count()
print(f'Number of GPUs   : {num_gpus}')
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    free_mem, total_mem = torch.cuda.mem_get_info(i)
    print(f'  GPU {i}: {props.name}')
    print(f'         Total VRAM : {total_mem / 1e9:.1f} GB')
    print(f'         Free  VRAM : {free_mem  / 1e9:.1f} GB')

DEVICE = '0,1' if num_gpus >= 2 else '0'
print(f'\nTraining device  : cuda:{DEVICE}')

## Phase 1 — Install Dependencies & Setup Paths

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'ultralytics>=8.2.0', 'albumentations>=1.4.0'], check=True)
print('Dependencies installed.')

In [ ]:
import os, shutil, yaml
from pathlib import Path

KAGGLE_INPUT   = Path('/kaggle/input')
WORKING_DIR    = Path('/kaggle/working')
CHECKPOINT_DIR = WORKING_DIR / 'checkpoints'
OUTPUT_DIR     = WORKING_DIR / 'outputs'

for d in [CHECKPOINT_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def find_dataset_dir(name='BoneFractureYolo8'):
    for candidate in KAGGLE_INPUT.rglob(name):
        if candidate.is_dir() and (candidate / 'train').exists():
            return candidate
    local = Path('data') / name
    if local.exists():
        return local
    raise FileNotFoundError(
        f'Could not find dataset folder "{name}" under {KAGGLE_INPUT}.\n'
        f'Contents of /kaggle/input:\n' +
        '\n'.join(str(p) for p in sorted(KAGGLE_INPUT.rglob('*'))[:40])
    )

EXTRACT_DIR = find_dataset_dir('BoneFractureYolo8')
print(f'Dataset found at: {EXTRACT_DIR}')

for split in ['train', 'valid', 'test']:
    img_dir = EXTRACT_DIR / split / 'images'
    lbl_dir = EXTRACT_DIR / split / 'labels'
    img_count = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
    lbl_count = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    print(f'  {split:5s}: {img_count} images, {lbl_count} labels')

## Phase 2 — Dataset Analysis & Class Balance Check

Checking class distribution is **critical** — imbalanced classes are a major cause of low mAP on rare classes like `elbow positive` and `wrist positive`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

CLASS_NAMES = [
    'elbow positive',
    'fingers positive',
    'forearm fracture',
    'humerus fracture',
    'humerus',
    'shoulder fracture',
    'wrist positive',
]

COLORS = plt.cm.Set1(np.linspace(0, 1, len(CLASS_NAMES)))

split_counts = {}
for split in ['train', 'valid', 'test']:
    lbl_dir = EXTRACT_DIR / split / 'labels'
    counter = Counter()
    for lbl_file in lbl_dir.glob('*.txt'):
        for line in lbl_file.read_text().strip().splitlines():
            if line.strip():
                counter[int(line.split()[0])] += 1
    split_counts[split] = counter
    print(f'\n{split.upper()} class counts:')
    for i, name in enumerate(CLASS_NAMES):
        bar = '█' * (counter.get(i, 0) // 10)
        print(f'  {name:20s}: {counter.get(i, 0):4d}  {bar}')

# Imbalance ratio
train_counts = split_counts['train']
max_cls = max(train_counts.values())
print('\nImbalance ratios (max / class_count):')
for i, name in enumerate(CLASS_NAMES):
    cnt = train_counts.get(i, 1)
    ratio = max_cls / cnt
    flag = ' ⚠️  SEVERELY IMBALANCED' if ratio > 5 else ''
    print(f'  {name:20s}: {ratio:.1f}x{flag}')

In [ ]:
DATA_YAML = WORKING_DIR / 'data.yaml'

data_config = {
    'path'  : str(EXTRACT_DIR),
    'train' : 'train/images',
    'val'   : 'valid/images',
    'test'  : 'test/images',
    'nc'    : 7,
    'names' : CLASS_NAMES,
}

with open(DATA_YAML, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print('data.yaml written to:', DATA_YAML)
print(DATA_YAML.read_text())

## Phase 3 — Full Training Run (Improved)

### Key improvements over v1:

| Parameter | v1 (bad) | v2 (fixed) | Why |
|-----------|----------|------------|-----|
| Model | YOLOv8m | **YOLOv8l** | More capacity for complex fracture patterns |
| Epochs | 50 | **300** | Model was still learning at epoch 34 |
| Patience | 10 | **50** | Avoid premature stopping |
| Optimizer | SGD (default) | **AdamW** | Better convergence on medical datasets |
| lr0 | 0.01 | **0.001** | SGD needs 0.01, AdamW needs lower |
| lrf | 0.01 | **0.01** | Cosine decay to lr0*lrf = 1e-5 |
| imgsz | 640 | **800** | Larger crops capture fine fracture details |
| Mosaic | 1.0 | **1.0** | Keep, but disable last 10 epochs |
| Mixup | 0.0 | **0.15** | Mix images to regularize |
| Copy-paste | 0.0 | **0.3** | Augment rare class instances |
| Scale | default | **0.9** | Random scale jitter |
| Perspective | 0.0 | **0.001** | Simulate X-ray angle variation |
| Label smoothing | 0.0 | **0.1** | Reduces overconfidence on rare classes |
| close_mosaic | N/A | **30** | Disable mosaic in last 30 epochs for stable convergence |

In [ ]:
from ultralytics import YOLO
import torch, yaml
from pathlib import Path

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Hyperparameters ─────────────────────────────────────────────────────────
MODEL    = 'yolov8l.pt'   # Larger model: yolov8l (43M params) vs yolov8m (25M)
EPOCHS   = 300
IMGSZ    = 800            # Higher resolution: captures small fracture lines better
BATCH    = -1             # -1 = auto-detect safe batch size (uses ~60% VRAM)
LR0      = 0.001          # AdamW works best at 1e-3
LRF      = 0.01           # Final LR = LR0 * LRF = 1e-5 (cosine schedule)
WARMUP_E = 5.0            # Warmup epochs
PATIENCE = 50             # Don't stop early before giving model a real chance
OPTIMIZER= 'AdamW'        # Better than SGD for this dataset size
WEIGHT_DECAY = 0.0005
MOMENTUM = 0.937
RUN_NAME = 'fracture_yolov8l_v2'

hparams = dict(
    model=MODEL, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    lr0=LR0, lrf=LRF, patience=PATIENCE, optimizer=OPTIMIZER,
    weight_decay=WEIGHT_DECAY, device=DEVICE, amp=True, seed=SEED
)
with open(OUTPUT_DIR / 'hparams_v2.yaml', 'w') as f:
    yaml.dump(hparams, f)
print('Hyperparameters saved.')

# Load YOLOv8l with COCO pre-trained weights
model = YOLO(MODEL)

print(f'Starting improved training on device: {DEVICE}')
print(f'Model        : {MODEL}')
print(f'Epochs       : {EPOCHS}  |  Patience: {PATIENCE}')
print(f'Optimizer    : {OPTIMIZER}  |  lr0: {LR0}  →  {LR0*LRF:.2e}')
print(f'Image size   : {IMGSZ}  |  Batch: {BATCH} (auto)')
print('=' * 60)

results = model.train(
    # ── Core ──────────────────────────────────────────────
    data        = str(DATA_YAML),
    epochs      = EPOCHS,
    imgsz       = IMGSZ,
    batch       = BATCH,
    device      = DEVICE,
    project     = str(WORKING_DIR / 'runs'),
    name        = RUN_NAME,
    exist_ok    = True,
    save        = True,
    save_period = 10,         # checkpoint every 10 epochs
    patience    = PATIENCE,
    seed        = SEED,
    verbose     = True,
    plots       = True,
    amp         = True,       # AMP: faster training

    # ── Optimizer ─────────────────────────────────────────
    optimizer   = OPTIMIZER,
    lr0         = LR0,
    lrf         = LRF,
    momentum    = MOMENTUM,
    weight_decay= WEIGHT_DECAY,
    warmup_epochs   = WARMUP_E,
    warmup_momentum = 0.8,
    warmup_bias_lr  = 0.1,
    cos_lr      = True,       # Cosine learning rate schedule

    # ── Loss weights ──────────────────────────────────────
    box         = 7.5,        # default
    cls         = 0.5,        # classification loss weight
    dfl         = 1.5,        # distribution focal loss
    label_smoothing = 0.1,    # prevents overconfidence on imbalanced classes

    # ── Augmentation (stronger than v1) ───────────────────
    mosaic      = 1.0,        # mosaic augmentation
    close_mosaic= 30,         # disable mosaic in last 30 epochs → stable convergence
    mixup       = 0.15,       # blend pairs of training images
    copy_paste  = 0.3,        # copy objects between images (helps rare classes)
    degrees     = 10.0,       # rotation ±10°
    translate   = 0.1,        # translation ±10%
    scale       = 0.9,        # random scale jitter
    shear       = 0.0,
    perspective = 0.001,      # mild perspective (X-ray angle variation)
    flipud      = 0.1,        # vertical flip (anatomical variation)
    fliplr      = 0.5,        # horizontal flip
    bgr         = 0.0,
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    erasing     = 0.4,        # random erasing (simulates X-ray artifacts)
    crop_fraction = 1.0,
)

BEST_MODEL = Path(results.save_dir) / 'weights' / 'best.pt'
print('\n=== Training Complete ===')
print(f'Best model   : {BEST_MODEL}')
print(f'Run dir      : {results.save_dir}')

## Phase 4 — Evaluation on Validation & Test Set

In [ ]:
from ultralytics import YOLO
from pathlib import Path

BEST_MODEL = WORKING_DIR / 'runs' / RUN_NAME / 'weights' / 'best.pt'
if not BEST_MODEL.exists():
    raise FileNotFoundError(f'best.pt not found at {BEST_MODEL}. Run Phase 3 first.')

best_model = YOLO(str(BEST_MODEL))

print('Evaluating best model on TEST set ...')
test_metrics = best_model.val(
    data    = str(DATA_YAML),
    split   = 'test',
    device  = DEVICE,
    imgsz   = IMGSZ,
    batch   = 8,  # use fixed batch for eval
    verbose = True,
    plots   = True,
    save_json = True,
    project = str(OUTPUT_DIR),
    name    = 'test_eval_v2',
)

print('\n=== TEST SET RESULTS ===')
print(f'mAP@50       : {test_metrics.box.map50:.4f}')
print(f'mAP@50-95    : {test_metrics.box.map:.4f}')
print(f'Precision    : {test_metrics.box.mp:.4f}')
print(f'Recall       : {test_metrics.box.mr:.4f}')
print()
print('Per-class AP@50:')
for i, (name, ap) in enumerate(zip(CLASS_NAMES, test_metrics.box.ap50)):
    flag = ' ⚠️' if ap < 0.3 else ' ✅' if ap > 0.5 else ''
    print(f'  {name:25s}: {ap:.4f}{flag}')

## Phase 5 — Results Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

RUN_DIR = WORKING_DIR / 'runs' / RUN_NAME

for plot_name, title, out_name in [
    ('results.png', 'Training & Validation Curves', 'training_curves_v2.png'),
    ('confusion_matrix_normalized.png', 'Confusion Matrix (Normalized)', 'confusion_matrix_v2.png'),
    ('PR_curve.png', 'Precision-Recall Curve', 'pr_curve_v2.png'),
    ('F1_curve.png', 'F1 Confidence Curve', 'f1_curve_v2.png'),
]:
    plot_path = RUN_DIR / plot_name
    if plot_path.exists():
        img = mpimg.imread(str(plot_path))
        plt.figure(figsize=(14, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title, fontsize=14)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / out_name, dpi=120, bbox_inches='tight')
        plt.show()
        print(f'Saved: {out_name}')
    else:
        print(f'{plot_name} not found at {plot_path}')

In [ ]:
import random
from pathlib import Path
import matplotlib.pyplot as plt

test_img_dir = EXTRACT_DIR / 'test' / 'images'
test_imgs = random.sample(list(test_img_dir.glob('*.jpg')), k=min(6, len(list(test_img_dir.glob('*.jpg')))))

preds = best_model.predict(
    source    = [str(p) for p in test_imgs],
    imgsz     = IMGSZ,
    device    = DEVICE,
    conf      = 0.25,
    iou       = 0.45,
    save      = False,
    verbose   = False,
)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, result in zip(axes, preds):
    img_bgr = result.plot()
    img_rgb = img_bgr[..., ::-1]
    ax.imshow(img_rgb)
    ax.axis('off')
    n_det = len(result.boxes) if result.boxes is not None else 0
    ax.set_title(f'{Path(result.path).name[:25]} | {n_det} det(s)', fontsize=8)

plt.suptitle('Model Predictions on Test Images (conf≥0.25)', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'test_predictions_v2.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: test_predictions_v2.png')

## Phase 6 — Summary

In [ ]:
print('=' * 60)
print('TRAINING SUMMARY v2')
print('=' * 60)
print(f'Model          : YOLOv8l')
print(f'Dataset        : BoneFractureYolo8 (7 classes)')
print(f'Optimizer      : AdamW  |  lr0={LR0}  →  {LR0*LRF:.2e}')
print(f'Augmentations  : mosaic + mixup + copy_paste + perspective')
print()
print('TEST SET METRICS:')
print(f'  mAP@50       : {test_metrics.box.map50:.4f}')
print(f'  mAP@50-95    : {test_metrics.box.map:.4f}')
print(f'  Precision    : {test_metrics.box.mp:.4f}')
print(f'  Recall       : {test_metrics.box.mr:.4f}')
print()
print('Per-class AP@50:')
for name, ap in zip(CLASS_NAMES, test_metrics.box.ap50):
    flag = ' ⚠️' if ap < 0.3 else ' ✅' if ap > 0.5 else ''
    print(f'  {name:25s}: {ap:.4f}{flag}')
print()
print('FILES TO DOWNLOAD from /kaggle/working/ :')
print(f'  best.pt          -> {BEST_MODEL}')
print(f'  hparams_v2.yaml  -> {OUTPUT_DIR}/hparams_v2.yaml')
print(f'  train_curves.png -> {OUTPUT_DIR}/training_curves_v2.png')
print(f'  conf_matrix.png  -> {OUTPUT_DIR}/confusion_matrix_v2.png')
print(f'  pr_curve.png     -> {OUTPUT_DIR}/pr_curve_v2.png')

## If mAP is Still Low — Next Steps

| Symptom | Fix |
|---------|-----|
| `elbow positive` AP < 0.3 | Dataset too small; try oversampling or class-weighted loss |
| Val loss still decreasing at epoch 300 | Train more epochs or use `patience=0` |
| Both train & val loss plateau | Try YOLOv8x or RTDETR-L (transformer-based) |
| Good mAP but bad per-class | Check label quality for worst-performing classes |
| CUDA OOM with batch=-1 | Set `batch=8` or `batch=4` explicitly |

### Optional: Two-Stage Training (Fine-tune on Hard Classes)

After v2 training, run a fine-tune pass with `copy_paste=0.8` and oversample the rare classes:
```python
model = YOLO('/kaggle/working/runs/fracture_yolov8l_v2/weights/best.pt')
model.train(
    data=str(DATA_YAML),
    epochs=50, lr0=0.0001, lrf=0.1,
    optimizer='AdamW', copy_paste=0.8, mixup=0.3,
    patience=20, name='fracture_yolov8l_finetune'
)
```